# 03 - Exploratory Data Analysis (EDA)
## Dataset Statistics, Label Balance & Sample Preview

Analyze the final translated datasets before training:
1. Label distribution across variants
2. Text length distributions
3. Sample previews in all 3 language variants

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import FINAL_DIR, INTERIM_DIR, FRIENDLY_LABELS

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)

## Step 1: Load All Datasets

In [ ]:
# Load all final datasets
variants = ['pure_urdu', 'mixed', 'roman_urdu']
splits = ['train', 'dev']

datasets = {}
for split in splits:
    for variant in variants:
        path = FINAL_DIR / f'{split}_{variant}.csv'
        if path.exists():
            df = pd.read_csv(path, encoding='utf-8-sig')
            datasets[f'{split}_{variant}'] = df
            print(f'  {split}_{variant}: {len(df):,} rows')
        else:
            print(f'  {split}_{variant}: NOT FOUND')

print(f'\nTotal datasets loaded: {len(datasets)}')

## Step 2: Label Distribution

In [ ]:
# Plot label distribution for each variant
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

label_names = {0: 'Supported', 1: 'Not Supported', 2: 'Not Enough Info'}

for idx, variant in enumerate(variants):
    key = f'train_{variant}'
    if key in datasets:
        df = datasets[key]
        counts = df['label'].value_counts().sort_index()
        labels = [label_names[i] for i in counts.index]
        
        axes[idx].bar(labels, counts.values, color=['#2ecc71', '#e74c3c', '#3498db'])
        axes[idx].set_title(f'Train - {variant}', fontsize=13)
        axes[idx].set_ylabel('Count')
        
        for j, v in enumerate(counts.values):
            axes[idx].text(j, v + 100, f'{v:,}', ha='center', fontsize=10)

plt.suptitle('Label Distribution by Variant', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3: Text Length Distribution

In [ ]:
# Token/word length distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, variant in enumerate(variants):
    key = f'train_{variant}'
    if key in datasets:
        df = datasets[key]
        claim_lens = df['claim'].astype(str).str.split().str.len()
        evidence_lens = df['evidence'].astype(str).str.split().str.len()
        
        axes[idx].hist(claim_lens, bins=50, alpha=0.7, label='Claim', color='#3498db')
        axes[idx].hist(evidence_lens, bins=50, alpha=0.7, label='Evidence', color='#e74c3c')
        axes[idx].set_title(f'{variant}', fontsize=13)
        axes[idx].set_xlabel('Word Count')
        axes[idx].legend()

plt.suptitle('Text Length Distribution (words)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4: Sample Preview - All 3 Variants

In [ ]:
# Side-by-side preview
n_samples = 5

for i in range(n_samples):
    print(f'\n{"=" * 70}')
    print(f'  SAMPLE {i+1}')
    print(f'{"=" * 70}')
    
    for variant in variants:
        key = f'train_{variant}'
        if key in datasets:
            row = datasets[key].iloc[i]
            label = label_names.get(row['label'], '?')
            print(f'\n  [{variant}] Label: {label}')
            print(f'    Claim:    {str(row["claim"])[:100]}')
            print(f'    Evidence: {str(row["evidence"])[:100]}')

## Step 5: Combined Dataset Stats

In [ ]:
# Summary table
summary_rows = []
for key, df in datasets.items():
    summary_rows.append({
        'Dataset': key,
        'Rows': len(df),
        'Supported': (df['label'] == 0).sum(),
        'Not Supported': (df['label'] == 1).sum(),
        'Not Enough Info': (df['label'] == 2).sum(),
        'Avg Claim Len': df['claim'].astype(str).str.len().mean().round(1),
        'Avg Evidence Len': df['evidence'].astype(str).str.len().mean().round(1),
        'Empty Evidence %': (df['evidence'].isna() | (df['evidence'] == '')).mean() * 100,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.style.format({
    'Rows': '{:,}',
    'Supported': '{:,}',
    'Not Supported': '{:,}',
    'Not Enough Info': '{:,}',
    'Empty Evidence %': '{:.1f}%',
})

## Step 6: Check for Data Quality Issues

In [ ]:
# Check for empty/null values
for key, df in datasets.items():
    nulls = df.isnull().sum()
    empty_claims = (df['claim'].astype(str).str.strip() == '').sum()
    empty_evidence = (df['evidence'].astype(str).str.strip() == '').sum()
    
    print(f'{key}:')
    print(f'  Null values:     {nulls.sum()}')
    print(f'  Empty claims:    {empty_claims}')
    print(f'  Empty evidence:  {empty_evidence}')
    print()